# Knockout strains: growth curves and endpoint titers

Figures 3B, 3C

Environment: `python_env` (`envs/python_env.yml`)

Inputs:
- `data/knockouts_OD600.xlsx`, `data/knockouts_plate_plan.xlsx`
- `data/knockouts_endpoint_titer.xlsx`

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, NullLocator
from scipy.stats import ttest_ind
from statannotations.Annotator import Annotator

plt.rcParams["pdf.fonttype"] = 42

ROOT = Path("..")
config = yaml.safe_load(open(ROOT / "config.yaml"))
OUT_DIR = Path(config["out_dir"])
FIG_DIR = OUT_DIR / "figures"
SRC_DIR = OUT_DIR / "source_data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

MS2_COLOR = "#FF7F0E"
CTRL_COLOR = "#1F77B4"
STRAIN_MARKERS = {"ΔlacY": "o", "ΔnhaR": "s", "ΔpgaA": "^"}

LABEL_SIZE = 30
LEGEND_TITLE_SIZE = 26
LEGEND_TEXT_SIZE = 24
TICK_SIZE = 28


def save_fig(fig, name, dpi, **kw):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight", facecolor="white", **kw)
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", facecolor="white", **kw)

In [ ]:
def read_plate(results_file, plan_file):
    plan = pd.read_excel(plan_file).rename(columns={"Unnamed: 0": "row"})
    plan = plan.melt(id_vars=["row"]).dropna()
    plan["well_id"] = plan["row"] + plan["variable"].astype(str)
    plan = plan.set_index("well_id")["value"].to_dict()

    od = pd.read_excel(results_file).rename(columns=plan)
    t = pd.to_datetime(od.Time, format="%H:%M:%S")
    od["Time"] = t.dt.hour + t.dt.minute / 60
    od = od.set_index("Time").drop(columns=["BLK"])

    df = od.melt(ignore_index=False).reset_index().rename(columns={"variable": "line"})
    df["strain"] = df.line.str.split("_").str[0]
    df["infection"] = df.line.str.split("_").str[1:].str.join("_")
    return df

## Figure 3B

In [ ]:
sns.set_context("notebook", font_scale=1.8)

od = read_plate(ROOT / "data" / "knockouts_OD600.xlsx", ROOT / "data" / "knockouts_plate_plan.xlsx")

strain_order = ["ΔlacY", "ΔnhaR", "ΔpgaA"]
infection_order = ["Control (LB)", "MS2 (MOI=10)"]
infection_palette = {"Control (LB)": CTRL_COLOR, "MS2 (MOI=10)": MS2_COLOR}
strain_dashes = {"ΔlacY": (1, 0), "ΔnhaR": (4, 2), "ΔpgaA": (1, 1.5)}
strain_marker_sizes = {"ΔlacY": 11, "ΔnhaR": 11, "ΔpgaA": 15}
line_width = 4
vlines = [1.75, 6.3]

d = od.copy()
d["strain"] = pd.Categorical(d.strain.str.replace("del-", "Δ", regex=False), categories=strain_order, ordered=True)
d["infection"] = pd.Categorical(d.infection.replace({"MOI10_MS2": "MS2 (MOI=10)", "LB": "Control (LB)"}),
                                categories=infection_order, ordered=True)
d = d.sort_values(["infection", "strain"])

fig, ax = plt.subplots(figsize=(10, 7.5))
sns.lineplot(data=d, x="Time", y="value", hue="infection", hue_order=infection_order,
             style="strain", style_order=strain_order, palette=infection_palette,
             dashes=strain_dashes, markers=STRAIN_MARKERS, markevery=0.1, markersize=11,
             markeredgecolor="white", markeredgewidth=1.2, linewidth=line_width,
             errorbar="sd", legend=False, ax=ax)

marker_to_size = {STRAIN_MARKERS[s]: strain_marker_sizes[s] for s in strain_order}
for line in ax.lines:
    if line.get_marker() in marker_to_size:
        line.set_markersize(marker_to_size[line.get_marker()])

ax.set_xlim((0, 12))
ax.set_xticks(range(0, 14, 2))
ax.grid(alpha=0.2)
ax.tick_params(axis="both", labelsize=TICK_SIZE)
ax.set_xlabel("Time (hours post infection)", fontsize=LABEL_SIZE)
ax.set_ylabel("OD$_{600}$", fontsize=LABEL_SIZE)
for x in vlines:
    ax.axvline(x, color="black", linestyle="--", linewidth=1.5, zorder=1)

infection_handles = [Line2D([0], [0], color=infection_palette[i], lw=line_width, label=i) for i in infection_order]
strain_handles = [
    Line2D([0], [0], color="black", lw=line_width,
           linestyle=(0, strain_dashes[s]) if strain_dashes[s] != (1, 0) else "-",
           marker=STRAIN_MARKERS[s], markersize=strain_marker_sizes[s],
           markeredgecolor="white", markeredgewidth=1.2, label=s)
    for s in strain_order
]
legend_kw = dict(loc="upper left", frameon=False, fontsize=LEGEND_TEXT_SIZE,
                 title_fontsize=LEGEND_TITLE_SIZE, alignment="left", handlelength=2.5)
leg1 = ax.legend(handles=infection_handles, title="Infection", bbox_to_anchor=(1.05, 1), **legend_kw)
ax.add_artist(leg1)
fig.canvas.draw()
y0 = leg1.get_window_extent().transformed(ax.transAxes.inverted()).y0
leg2 = ax.legend(handles=strain_handles, title="Strain", bbox_to_anchor=(1.05, y0), **legend_kw)

save_fig(fig, "Fig3B", 800, bbox_extra_artists=[leg1, leg2])
plt.show()
d.rename(columns={"Time": "time_h", "value": "OD600"}).to_csv(SRC_DIR / "Fig3B_data.csv", index=False)

## Figure 3C

In [ ]:
sns.set_theme(style="ticks")

df = pd.read_excel(ROOT / "data" / "knockouts_endpoint_titer.xlsx")
df["conc"] = pd.to_numeric(df["conc"], errors="coerce")
df["MOI"] = df["MOI"].astype(float).apply(lambda x: f"{x:g}")
df = df.dropna(subset=["conc"])
df["strain"] = df["strain"].replace({"pgaA": "ΔpgaA", "lacY": "ΔlacY"})

control_means = df[df["strain"] == "ΔlacY"].groupby("MOI")["conc"].mean()
df["EOP"] = df["conc"] / df["MOI"].map(control_means)

moi_order = ["0.1", "10"]
strain_order = ["ΔlacY", "ΔpgaA"]
plot_params = {"data": df, "x": "MOI", "y": "EOP", "hue": "strain", "order": moi_order,
               "hue_order": strain_order, "dodge": True, "palette": {s: MS2_COLOR for s in strain_order}}

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.set_yscale("log")

for strain in strain_order:
    sdf = df[df["strain"] == strain]
    sns.stripplot(**{**plot_params, "data": sdf}, marker=STRAIN_MARKERS[strain], size=10, alpha=0.45,
                  jitter=False, edgecolor="white", linewidth=1, legend=False, zorder=2, ax=ax)
    n_before = len(ax.lines)
    sns.pointplot(**{**plot_params, "data": sdf, "dodge": 0.4}, markers=STRAIN_MARKERS[strain], markersize=18,
                  linestyle="none", capsize=0.1, errorbar="sd", err_kws={"linewidth": 4},
                  legend=False, ax=ax)
    for line in ax.lines[n_before:]:
        line.set_zorder(4)
        if line.get_marker() not in ("", "None", None):
            line.set_markeredgecolor("white")
            line.set_markeredgewidth(1.5)

pairs = [(("10", "ΔlacY"), ("10", "ΔpgaA")), (("0.1", "ΔlacY"), ("0.1", "ΔpgaA"))]
annotator = Annotator(ax, pairs, **plot_params)
annotator.configure(test="t-test_ind", text_format="star", loc="inside",
                    hide_non_significant=True, fontsize=TICK_SIZE, line_width=2)
annotator.apply_and_annotate()

ax.set_ylabel("Relative endpoint titer", fontsize=LABEL_SIZE, labelpad=10)
ax.set_xlabel("MOI", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylim(0.5, 70)
ax.set_yticks([1, 2, 5, 10, 20, 50])
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.yaxis.set_minor_locator(NullLocator())
ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.tick_params(axis="both", which="major", labelsize=TICK_SIZE, colors="black", length=7, width=1.5)
ax.axhline(1, color="black", linestyle="--", linewidth=2, alpha=0.7)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.5)
    spine.set_color("black")

handles = [Line2D([0], [0], color=MS2_COLOR, linestyle="none", marker=STRAIN_MARKERS[s], markersize=18,
                  markeredgecolor="white", markeredgewidth=1.5, label=s) for s in strain_order]
ax.legend(handles=handles, title="Strain", bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False,
          fontsize=LEGEND_TEXT_SIZE, title_fontsize=LEGEND_TITLE_SIZE, labelspacing=0.45, alignment="left")

plt.tight_layout()
save_fig(fig, "Fig3C", 600)
plt.show()

stats = pd.DataFrame([
    {"MOI": m, "n_lacY": len(a), "n_pgaA": len(b), "t": ttest_ind(a, b).statistic, "p": ttest_ind(a, b).pvalue}
    for m in moi_order
    for a, b in [(df[(df.MOI == m) & (df.strain == "ΔlacY")].EOP, df[(df.MOI == m) & (df.strain == "ΔpgaA")].EOP)]
])
df.to_csv(SRC_DIR / "Fig3C_data.csv", index=False)
stats.to_csv(SRC_DIR / "Fig3C_stats.csv", index=False)